In [54]:
import pandas as pd

In [55]:
df =pd.read_csv("IMDB Dataset.csv")

In [56]:
df.shape # for dataset structure

(50000, 2)

In [57]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [58]:
df.isnull().sum() # check for null values

review       0
sentiment    0
dtype: int64

In [59]:
df.drop_duplicates(inplace = True) # drop or remove duplicate values from dataset

In [60]:
df.shape

(49582, 2)

## Pre processing


In [61]:
# convert the text to lower case
# remove urls such https , http
# remove html tages
# remove punctuation (, ? / , #)
# remove stopwords (a , and , or , the , is )
# stemming -  convert word to actal noun word
# encode sentiments 
# vectorization - means convert all text to numbers for (TF _IDF)


#### 1.converting to lowercase

In [62]:
df["review"] = df["review"].str.lower()

#### 2. Removing the URLs

In [63]:
import re

#example
# sample_txt = "abc is the word , abc" # abc ==> xyz

# re.sub("abc","xyz",sample_txt)

In [64]:
def remove_urls(text):
    text = re.sub(r"http\S+" ,"",text) #pattern , replacement, string   eg = https://www.google.com
    return text
df["review"] = df["review"].apply(remove_urls)

#### 3. Remove Punctuations

In [65]:
def remove_punctuations(text):
    text = re.sub(r"^[A-Za-z0-9\s]" ,"",text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

#### 4. Removing HTMLS Tags

In [66]:
def remove_html(text):
    text = re.sub(r"<.*?>" ,"",text) # . => any charater , * => 0 or more character
    return text

df["review"] = df["review"].apply(remove_html)

#### 5. Removing the stopwords

In [67]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [68]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [69]:
# sample_text = "I like coding in python"
# tokens = word_tokenize(sample_text)

In [70]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [71]:
df.head()

,review,sentiment
0,ne revewers nted wtchg 1 oz epode 'll ...,positive
1,wderful ltle producti. filming technique u...,positive
2,thought wderful wy spend tme o hot summ...,positive
3,sclly 's fmly lttle boy (jke) thks 's zomb...,negative
4,"etter mttei's ""love time mey"" vully stunn...",positive


#### 6. Stemming ==> base form . ex = running -> run

In [72]:
# running => run
# played => play
# porterSteming

from nltk.stem import PorterStemmer

In [74]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [75]:
df.head()

,review,sentiment
0,ne revew nted wtchg 1 oz epod 'll hook . y rgh...,positive
1,wder ltle producti . film techniqu unssuming- ...,positive
2,thought wder wy spend tme o hot summer weekend...,positive
3,sclli 's fmli lttle boy ( jke ) thk 's zomb cl...,negative
4,etter mttei 's `` love time mey '' vulli stunn...,positive


#### 7. Encoding

In [79]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [80]:
y = df["sentiment"]

In [82]:
df.head()

,review,sentiment
0,ne revew nted wtchg 1 oz epod 'll hook . y rgh...,1
1,wder ltle producti . film techniqu unssuming- ...,1
2,thought wder wy spend tme o hot summer weekend...,1
3,sclli 's fmli lttle boy ( jke ) thk 's zomb cl...,0
4,etter mttei 's `` love time mey '' vulli stunn...,1


#### 8 . Vectorization

In [83]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000) 
X = tf.fit_transform(df["review"])

In [84]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4058804 stored elements and shape (49582, 5000)>

## Dataset & DataLoaders

In [100]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size =0.2 , random_state = 42
)

In [101]:
X_train.shape

(39665, 5000)

In [102]:
X_test.shape

(9917, 5000)

In [106]:
import torch
from torch.utils.data import TensorDataset, DataLoader


In [104]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [107]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
    
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
    
)

In [109]:
train_loader = DataLoader(train_set,shuffle= True,batch_size=64)
test_loader = DataLoader(test_set,shuffle= True,batch_size=64)

## Build our RNN

In [112]:
import torch.nn as nn
import torch.optim as optim

In [113]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)


        #Fully connected layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        # optional => shape(num of layers , batch size , hidden size)
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_ = self.rnn(x,h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep


        out = self.fc(out[:, -1, :])
        return out
        

In [114]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the model RNN

In [118]:
# unsqueeze squeeze
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) ## add singleton direction
        
        outputs = model(Xb) # (batch_size,1)

        outputs = torch.sigmoid(outputs.squeeze()) #(batch_size,)

        loss = criterion(outputs,yb) # compute loss
        loss.backward() # back prop
        optimizer.step() # weights update

    print(f"{epoch+1}/{epochs} and loss = {loss.item()}")

1/10 and loss = 0.4398544728755951
2/10 and loss = 0.22096441686153412
3/10 and loss = 0.1462002843618393
4/10 and loss = 0.3786168396472931
5/10 and loss = 0.3108103573322296
6/10 and loss = 0.10457604378461838
7/10 and loss = 0.4120825231075287
8/10 and loss = 0.22102618217468262
9/10 and loss = 0.306607186794281
10/10 and loss = 0.27900901436805725


In [121]:
# evaluate 

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb,yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals +=yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.30805687203792
